# 21_strong_model_residual_phm_diagnostics.ipynb

NB21 defines a controlled strong-model residual diagnostic validation after NB20. NB20 used the weaker canonical `Baseline_Prediction` source to construct forecasting-based PHM indicators. NB21 tests whether the same residual diagnostic layer remains informative and becomes more selective when residuals are produced by a stronger leakage-safe forecasting model.

The model is used only as a residual source for diagnostics. This notebook does not introduce a new canonical benchmark, does not perform architecture search, and does not update benchmark artifacts.

Leakage controls:

- Fit the model only on `train_final.csv`.
- Fit imputers and encoders only on the training split.
- Use validation residuals only for threshold calibration and any configuration decision.
- Evaluate test once only after the validation threshold policy is fixed.
- Do not update `baseline_metrics.csv`, `requirements.txt`, or NB20 diagnostic outputs.

## Run Controls

The committed scaffold uses safe defaults:

```python
SMOKE_MODE = True
RUN_FULL_DIAGNOSTICS = False
EXPORT_RESULTS = False
FULL_EXPORT_RESIDUAL_RECORDS = False
RANDOM_STATE = 42
```

A full local run should explicitly switch to:

```python
SMOKE_MODE = False
RUN_FULL_DIAGNOSTICS = True
EXPORT_RESULTS = True
FULL_EXPORT_RESIDUAL_RECORDS = False
```

Full residual-row exports remain disabled unless `FULL_EXPORT_RESIDUAL_RECORDS=True` is set intentionally.

In [ ]:
from pathlib import Path

SMOKE_MODE = True
RUN_FULL_DIAGNOSTICS = False
EXPORT_RESULTS = False
FULL_EXPORT_RESIDUAL_RECORDS = False
RANDOM_STATE = 42

TARGET_COL = 'Power_Output_Normalized'
PARK_COL = 'park_id'
TIME_COL = 'timestamp'

DATA_DIR = Path('data/processed')
TRAIN_PATH = DATA_DIR / 'train_final.csv'
VAL_PATH = DATA_DIR / 'val_final.csv'
TEST_PATH = DATA_DIR / 'test_final.csv'
OUTPUT_DIR = DATA_DIR / 'diagnostics' / 'strong_model_residual_phm'

SMOKE_MAX_PARKS = 8
SMOKE_MAX_ROWS_PER_PARK = 240
MIN_VALIDATION_ROWS_PER_PARK = 48
EPSILON = 1e-9

PROHIBITED_WRITE_PATHS = {
    DATA_DIR / 'baseline_metrics.csv',
    Path('requirements.txt'),
    Path('notebooks/20_residual_phm_diagnostics.ipynb'),
    Path('docs/RESIDUAL_PHM_DIAGNOSTICS_AUDIT.md'),
}

PREDICTION_COLUMN_CANDIDATES = {
    'Baseline_Prediction',
    'prediction',
    'y_pred',
    'predicted',
    'model_prediction',
}

DIAGNOSTIC_LEAKAGE_TERMS = {
    'residual',
    'abs_error',
    'squared_error',
    'error',
    'warning',
    'warning_flag',
    'warning_reason',
    'y_true',
    'y_pred',
    'prediction_source',
    'diagnostic',
    'threshold',
    'z_score',
    'robust_z',
}

EXPORT_PREFIX = 'strong_residual_phm'

USE_PARK_ID_AS_FEATURE = True

In [ ]:
import json
import math
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

pd.set_option('display.max_columns', 120)
warnings.filterwarnings('ignore', category=FutureWarning)

## Split And Path Audit

The notebook uses the existing processed canonical splits only. No benchmark file is overwritten.

In [ ]:
split_paths = {
    'train': TRAIN_PATH,
    'validation': VAL_PATH,
    'test': TEST_PATH,
}

path_audit_df = pd.DataFrame(
    [
        {
            'split': split_name,
            'path': str(path),
            'exists': path.exists(),
            'size_mb': path.stat().st_size / (1024**2) if path.exists() else np.nan,
        }
        for split_name, path in split_paths.items()
    ]
)

missing_paths = path_audit_df.loc[~path_audit_df['exists'], 'path'].tolist()
if missing_paths:
    raise FileNotFoundError(f'Missing required split files: {missing_paths}')

path_audit_df

In [ ]:
def load_split(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={PARK_COL: 'string'})
    if PARK_COL in df.columns:
        df[PARK_COL] = df[PARK_COL].astype('string')
    if TIME_COL in df.columns:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce')
    return df


train_df = load_split(TRAIN_PATH)
val_df = load_split(VAL_PATH)
test_df = load_split(TEST_PATH)

split_shape_audit_df = pd.DataFrame(
    [
        {'split': 'train', 'rows': len(train_df), 'columns': train_df.shape[1]},
        {'split': 'validation', 'rows': len(val_df), 'columns': val_df.shape[1]},
        {'split': 'test', 'rows': len(test_df), 'columns': test_df.shape[1]},
    ]
)

split_shape_audit_df

## Split Temporal Audit

This audit reports split-level temporal coverage and order sanity only. It does not change model configuration, smoke sampling, or threshold derivation.

In [ ]:
def split_temporal_audit(split_name: str, df: pd.DataFrame) -> dict:
    timestamps = pd.to_datetime(df[TIME_COL], errors='coerce')
    return {
        'split': split_name,
        'rows': len(df),
        'parks': df[PARK_COL].nunique(dropna=True),
        'timestamp_min': timestamps.min(),
        'timestamp_max': timestamps.max(),
        'missing_timestamp_count': int(timestamps.isna().sum()),
    }


def temporal_order_sanity(split_name: str, df: pd.DataFrame) -> dict:
    ordered = df[[PARK_COL, TIME_COL]].copy()
    ordered[TIME_COL] = pd.to_datetime(ordered[TIME_COL], errors='coerce')
    per_park_monotonic = ordered.groupby(PARK_COL)[TIME_COL].apply(lambda s: bool(s.is_monotonic_increasing))
    return {
        'split': split_name,
        'parks_checked': int(per_park_monotonic.shape[0]),
        'parks_temporally_monotonic': int(per_park_monotonic.sum()),
        'parks_with_temporal_order_issue': int((~per_park_monotonic).sum()),
        'all_parks_temporally_monotonic': bool(per_park_monotonic.all()) if len(per_park_monotonic) else False,
    }


split_temporal_audit_df = pd.DataFrame(
    [
        split_temporal_audit('train', train_df),
        split_temporal_audit('validation', val_df),
        split_temporal_audit('test', test_df),
    ]
)

temporal_order_sanity_df = pd.DataFrame(
    [
        temporal_order_sanity('train', train_df),
        temporal_order_sanity('validation', val_df),
        temporal_order_sanity('test', test_df),
    ]
)

split_temporal_audit_df.merge(temporal_order_sanity_df, on='split', how='left')

## Required-Column Audit

In [ ]:
REQUIRED_COLUMNS = [TARGET_COL, PARK_COL, TIME_COL]

def required_column_audit(split_name: str, df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {'split': split_name, 'column': column, 'present': column in df.columns}
            for column in REQUIRED_COLUMNS
        ]
    )


required_column_audit_df = pd.concat(
    [
        required_column_audit('train', train_df),
        required_column_audit('validation', val_df),
        required_column_audit('test', test_df),
    ],
    ignore_index=True,
)

missing_required = required_column_audit_df.loc[~required_column_audit_df['present']]
if not missing_required.empty:
    raise ValueError(missing_required.to_string(index=False))

required_column_audit_df

## Feature Schema Audit

Direct prediction columns, diagnostic columns, leakage-like columns, the target, and raw timestamp are excluded from the strong-model feature set. Feature selection uses train and validation schema only; test schema is audited for availability but does not select features.

In [ ]:
PREDICTION_CANDIDATES_LOWER = {candidate.lower() for candidate in PREDICTION_COLUMN_CANDIDATES}


def is_prediction_like_column(column: str) -> bool:
    lower = str(column).lower()
    return lower in PREDICTION_CANDIDATES_LOWER or lower.endswith('_prediction') or lower.endswith('_pred')


def is_diagnostic_leakage_like_column(column: str) -> bool:
    lower = str(column).lower()
    return any(term in lower for term in DIAGNOSTIC_LEAKAGE_TERMS)


def feature_exclusion_reasons(column: str, train: pd.DataFrame, validation: pd.DataFrame) -> list[str]:
    reasons = []
    if column == TARGET_COL:
        reasons.append('target')
    if column == TIME_COL:
        reasons.append('timestamp_not_model_feature')
    if column == PARK_COL and not USE_PARK_ID_AS_FEATURE:
        reasons.append('park_id_disabled_as_feature')
    if column not in validation.columns:
        reasons.append('missing_from_validation')
    if is_prediction_like_column(column):
        reasons.append('prediction_like_column')
    if is_diagnostic_leakage_like_column(column):
        reasons.append('diagnostic_or_leakage_like_column')
    return reasons


def select_feature_columns(train: pd.DataFrame, validation: pd.DataFrame, test: pd.DataFrame) -> tuple[list[str], pd.DataFrame]:
    feature_columns = []
    audit_rows = []
    for column in train.columns:
        reasons = feature_exclusion_reasons(column, train, validation)
        used_as_feature = not reasons
        if used_as_feature:
            feature_columns.append(column)
        audit_rows.append(
            {
                'column': column,
                'in_train': column in train.columns,
                'in_validation': column in validation.columns,
                'in_test': column in test.columns,
                'used_as_feature': used_as_feature,
                'excluded_reason': 'used' if used_as_feature else ';'.join(reasons),
                'test_available_for_full_run': column in test.columns,
            }
        )
    audit_df = pd.DataFrame(audit_rows)
    return feature_columns, audit_df


def assert_full_run_feature_availability(test: pd.DataFrame, columns: list[str]) -> None:
    missing = [column for column in columns if column not in test.columns]
    if missing:
        raise ValueError(f'Full diagnostics require test feature columns that are missing: {missing}')


feature_columns, feature_schema_audit_df = select_feature_columns(train_df, val_df, test_df)
if not feature_columns:
    raise ValueError('No usable leakage-safe feature columns were found.')

feature_schema_audit_df

## Smoke-Mode Row Limiting

Smoke mode keeps a small, temporally ordered subset from train/validation shared parks when possible. Test data does not influence smoke selection and is not included in default diagnostic setup when full diagnostics are disabled.

In [ ]:
def choose_smoke_parks(train: pd.DataFrame, validation: pd.DataFrame) -> list[str]:
    train_parks = set(train[PARK_COL].dropna().astype('string'))
    validation_parks = set(validation[PARK_COL].dropna().astype('string'))
    shared_parks = train_parks & validation_parks
    candidate_parks = shared_parks if shared_parks else train_parks
    counts = train.loc[train[PARK_COL].isin(candidate_parks), PARK_COL].value_counts()
    return counts.head(SMOKE_MAX_PARKS).index.astype('string').tolist()


def limit_smoke_split(df: pd.DataFrame, smoke_parks: list[str]) -> pd.DataFrame:
    ordered = df.loc[df[PARK_COL].isin(smoke_parks)].sort_values([PARK_COL, TIME_COL])
    limited = ordered.groupby(PARK_COL, group_keys=False, sort=False).head(SMOKE_MAX_ROWS_PER_PARK)
    return limited.reset_index(drop=True)


if SMOKE_MODE:
    smoke_parks = choose_smoke_parks(train_df, val_df)
    train_work_df = limit_smoke_split(train_df, smoke_parks)
    val_work_df = limit_smoke_split(val_df, smoke_parks)
    test_work_df = limit_smoke_split(test_df, smoke_parks) if RUN_FULL_DIAGNOSTICS else test_df.iloc[0:0].copy()
else:
    smoke_parks = []
    train_work_df = train_df.copy()
    val_work_df = val_df.copy()
    test_work_df = test_df.copy() if RUN_FULL_DIAGNOSTICS else test_df.iloc[0:0].copy()

smoke_audit_df = pd.DataFrame(
    [
        {'split': 'train', 'rows': len(train_work_df), 'parks': train_work_df[PARK_COL].nunique(), 'selection_role': 'model_fit'},
        {'split': 'validation', 'rows': len(val_work_df), 'parks': val_work_df[PARK_COL].nunique(), 'selection_role': 'threshold_calibration'},
        {'split': 'test', 'rows': len(test_work_df), 'parks': test_work_df[PARK_COL].nunique(), 'selection_role': 'held_out_until_full_diagnostics'},
    ]
)

smoke_audit_df

## Train-Only Preprocessing Setup

In [ ]:
def split_feature_types(train: pd.DataFrame, columns: list[str]) -> tuple[list[str], list[str]]:
    categorical_columns = []
    numeric_columns = []
    for column in columns:
        dtype = train[column].dtype
        if column == PARK_COL or pd.api.types.is_object_dtype(dtype) or pd.api.types.is_string_dtype(dtype) or pd.api.types.is_categorical_dtype(dtype):
            categorical_columns.append(column)
        else:
            numeric_columns.append(column)
    return numeric_columns, categorical_columns


def build_train_only_preprocessor(train: pd.DataFrame, columns: list[str]) -> tuple[ColumnTransformer, pd.DataFrame]:
    numeric_columns, categorical_columns = split_feature_types(train, columns)
    transformers = []
    if numeric_columns:
        transformers.append(('numeric', SimpleImputer(strategy='median'), numeric_columns))
    if categorical_columns:
        categorical_pipeline = Pipeline(
            steps=[
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
            ]
        )
        transformers.append(('categorical', categorical_pipeline, categorical_columns))
    if not transformers:
        raise ValueError('No preprocessing transformers could be built.')

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
    preprocessing_audit_df = pd.DataFrame(
        [
            {'feature_family': 'numeric', 'count': len(numeric_columns), 'columns': numeric_columns},
            {'feature_family': 'categorical', 'count': len(categorical_columns), 'columns': categorical_columns},
        ]
    )
    return preprocessor, preprocessing_audit_df


preprocessor_template, preprocessing_audit_df = build_train_only_preprocessor(train_work_df, feature_columns)
preprocessing_audit_df

## Strong-Model Selection Policy

The preferred residual model is `XGBRegressor` when it imports successfully in the current environment. If unavailable, the controlled fallback is `HistGradientBoostingRegressor`. `RandomForestRegressor` is reserved as an emergency fallback. No package installation and no hyperparameter search are performed.

In [ ]:
def select_strong_model(random_state: int):
    try:
        from xgboost import XGBRegressor

        model = XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            objective='reg:squarederror',
            eval_metric='rmse',
            tree_method='hist',
            n_jobs=-1,
            random_state=random_state,
        )
        policy = {'selected_model': 'XGBRegressor', 'fallback_reason': None}
        return model, policy
    except Exception as xgb_error:
        try:
            model = HistGradientBoostingRegressor(
                max_iter=250,
                learning_rate=0.05,
                l2_regularization=0.01,
                early_stopping=False,
                random_state=random_state,
            )
            policy = {
                'selected_model': 'HistGradientBoostingRegressor',
                'fallback_reason': f'XGBoost unavailable: {type(xgb_error).__name__}',
            }
            return model, policy
        except Exception as histogram_error:
            model = RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                n_jobs=-1,
                random_state=random_state,
            )
            policy = {
                'selected_model': 'RandomForestRegressor',
                'fallback_reason': f'Gradient boosting unavailable: {type(histogram_error).__name__}',
            }
            return model, policy


model_template, model_policy = select_strong_model(RANDOM_STATE)
model_policy_df = pd.DataFrame([model_policy])
model_policy_df

## Model Fitting And Prediction Functions

In [ ]:
def fit_residual_source_model(train: pd.DataFrame, columns: list[str], random_state: int) -> dict:
    preprocessor, preprocessing_audit = build_train_only_preprocessor(train, columns)
    model, policy = select_strong_model(random_state)
    pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model),
        ]
    )
    y_train = train[TARGET_COL].astype(float)
    pipeline.fit(train[columns], y_train)
    return {
        'pipeline': pipeline,
        'feature_columns': columns,
        'model_policy': policy,
        'preprocessing_audit': preprocessing_audit,
    }


def predict_residual_source(artifacts: dict, df: pd.DataFrame) -> np.ndarray:
    columns = artifacts['feature_columns']
    return np.asarray(artifacts['pipeline'].predict(df[columns]), dtype=float)


model_artifacts = fit_residual_source_model(train_work_df, feature_columns, RANDOM_STATE)
train_predictions = predict_residual_source(model_artifacts, train_work_df)
validation_predictions = predict_residual_source(model_artifacts, val_work_df)

test_predictions = None
TEST_EVALUATION_COUNT = 0

pd.DataFrame([model_artifacts['model_policy']])

## Metrics Function

In [ ]:
def regression_metrics(y_true, y_pred) -> dict:
    y_true_array = np.asarray(y_true, dtype=float)
    y_pred_array = np.asarray(y_pred, dtype=float)
    valid_mask = np.isfinite(y_true_array) & np.isfinite(y_pred_array)
    if valid_mask.sum() == 0:
        return {'MAE': np.nan, 'RMSE': np.nan, 'R2': np.nan, 'rows': 0}
    y_true_valid = y_true_array[valid_mask]
    y_pred_valid = y_pred_array[valid_mask]
    mse = mean_squared_error(y_true_valid, y_pred_valid)
    return {
        'MAE': float(mean_absolute_error(y_true_valid, y_pred_valid)),
        'RMSE': float(math.sqrt(mse)),
        'R2': float(r2_score(y_true_valid, y_pred_valid)) if len(y_true_valid) > 1 else np.nan,
        'rows': int(valid_mask.sum()),
    }


def build_model_fit_audit(train: pd.DataFrame, validation: pd.DataFrame, test: pd.DataFrame | None, train_pred, validation_pred, test_pred) -> pd.DataFrame:
    rows = []
    for split_name, split_df, predictions in [
        ('train', train, train_pred),
        ('validation', validation, validation_pred),
        ('test', test, test_pred),
    ]:
        if split_df is None or predictions is None:
            rows.append({'split': split_name, 'MAE': np.nan, 'RMSE': np.nan, 'R2': np.nan, 'rows': 0, 'status': 'not_evaluated'})
        else:
            metrics = regression_metrics(split_df[TARGET_COL], predictions)
            metrics.update({'split': split_name, 'status': 'evaluated'})
            rows.append(metrics)
    audit = pd.DataFrame(rows)
    train_mae = audit.loc[audit['split'].eq('train'), 'MAE'].iloc[0]
    validation_mae = audit.loc[audit['split'].eq('validation'), 'MAE'].iloc[0]
    test_mae = audit.loc[audit['split'].eq('test'), 'MAE'].iloc[0]
    audit['train_validation_MAE_gap'] = validation_mae - train_mae if np.isfinite(train_mae) and np.isfinite(validation_mae) else np.nan
    audit['validation_test_MAE_gap'] = test_mae - validation_mae if np.isfinite(validation_mae) and np.isfinite(test_mae) else np.nan
    return audit[['split', 'rows', 'MAE', 'RMSE', 'R2', 'train_validation_MAE_gap', 'validation_test_MAE_gap', 'status']]


model_fit_audit_df = build_model_fit_audit(train_work_df, val_work_df, None, train_predictions, validation_predictions, None)
model_fit_audit_df

## Validation-Derived Threshold Policy

All warning thresholds are calibrated on validation residuals. Per-park thresholds are used where validation support is sufficient; sparse parks fall back to global validation thresholds.

In [ ]:
def base_residual_frame(df: pd.DataFrame, predictions, split_name: str) -> pd.DataFrame:
    records = df[[PARK_COL, TIME_COL, TARGET_COL]].copy()
    records['split'] = split_name
    records['y_true'] = records[TARGET_COL].astype(float)
    records['y_pred'] = np.asarray(predictions, dtype=float)
    records['residual'] = records['y_true'] - records['y_pred']
    records['abs_error'] = records['residual'].abs()
    records['squared_error'] = records['residual'] ** 2
    records['residual_sign'] = np.select(
        [records['residual'].gt(0), records['residual'].lt(0)],
        ['observed_above_prediction', 'observed_below_prediction'],
        default='zero',
    )
    return records.sort_values([PARK_COL, TIME_COL]).reset_index(drop=True)


def add_rolling_residual_indicators(records: pd.DataFrame) -> pd.DataFrame:
    records = records.sort_values([PARK_COL, TIME_COL]).reset_index(drop=True)
    grouped = records.groupby(PARK_COL, group_keys=False, sort=False)
    records['rolling_MAE_24'] = grouped['abs_error'].transform(lambda s: s.rolling(24, min_periods=1).mean())
    records['rolling_MAE_72'] = grouped['abs_error'].transform(lambda s: s.rolling(72, min_periods=1).mean())
    records['rolling_bias_24'] = grouped['residual'].transform(lambda s: s.rolling(24, min_periods=1).mean())
    records['rolling_bias_72'] = grouped['residual'].transform(lambda s: s.rolling(72, min_periods=1).mean())
    return records


def derive_validation_thresholds(validation: pd.DataFrame, predictions) -> dict:
    validation_records = add_rolling_residual_indicators(base_residual_frame(validation, predictions, 'validation'))
    residual = validation_records['residual'].dropna()
    abs_error = validation_records['abs_error'].dropna()
    rolling_mae_24 = validation_records['rolling_MAE_24'].dropna()
    residual_median = float(residual.median())
    residual_mad = float(np.median(np.abs(residual - residual_median))) if len(residual) else np.nan

    global_thresholds = {
        'residual_mean': float(residual.mean()),
        'residual_std': float(residual.std(ddof=0)),
        'abs_error_q90': float(abs_error.quantile(0.90)),
        'abs_error_q95': float(abs_error.quantile(0.95)),
        'abs_error_q99': float(abs_error.quantile(0.99)),
        'residual_median': residual_median,
        'residual_mad': residual_mad,
        'rolling_MAE_24_q95': float(rolling_mae_24.quantile(0.95)),
    }

    per_park_thresholds = (
        validation_records.groupby(PARK_COL)
        .agg(
            validation_rows=('abs_error', 'size'),
            abs_error_q95=('abs_error', lambda s: float(s.quantile(0.95))),
            rolling_MAE_24_q95=('rolling_MAE_24', lambda s: float(s.quantile(0.95))),
        )
        .reset_index()
    )
    per_park_thresholds = per_park_thresholds.loc[per_park_thresholds['validation_rows'] >= MIN_VALIDATION_ROWS_PER_PARK]

    return {
        'global': global_thresholds,
        'per_park': per_park_thresholds,
        'validation_threshold_source_rows': len(validation_records),
    }


thresholds = derive_validation_thresholds(val_work_df, validation_predictions)
threshold_global_df = pd.DataFrame([thresholds['global']])
threshold_per_park_df = thresholds['per_park']
threshold_global_df

## Residual Record Construction And Warning Rules

In [ ]:
RESIDUAL_RECORD_COLUMNS = [
    'y_true',
    'y_pred',
    'residual',
    'abs_error',
    'squared_error',
    'residual_sign',
    'residual_z_score',
    'robust_residual_z_score',
    'rolling_MAE_24',
    'rolling_MAE_72',
    'rolling_bias_24',
    'rolling_bias_72',
    'warning_abs_q95',
    'warning_robust_z',
    'warning_rolling_mae_24',
    'warning_flag',
    'warning_reason',
]


def construct_residual_records(df: pd.DataFrame, predictions, split_name: str, threshold_bundle: dict) -> pd.DataFrame:
    records = add_rolling_residual_indicators(base_residual_frame(df, predictions, split_name))
    global_thresholds = threshold_bundle['global']
    residual_std = max(abs(global_thresholds['residual_std']), EPSILON)
    robust_scale = max(1.4826 * abs(global_thresholds['residual_mad']), EPSILON)

    records['residual_z_score'] = (records['residual'] - global_thresholds['residual_mean']) / residual_std
    records['robust_residual_z_score'] = (records['residual'] - global_thresholds['residual_median']) / robust_scale

    per_park = threshold_bundle['per_park'].set_index(PARK_COL) if not threshold_bundle['per_park'].empty else pd.DataFrame()
    if not per_park.empty:
        records['abs_q95_threshold'] = records[PARK_COL].map(per_park['abs_error_q95']).fillna(global_thresholds['abs_error_q95'])
        records['rolling_mae_24_q95_threshold'] = records[PARK_COL].map(per_park['rolling_MAE_24_q95']).fillna(global_thresholds['rolling_MAE_24_q95'])
    else:
        records['abs_q95_threshold'] = global_thresholds['abs_error_q95']
        records['rolling_mae_24_q95_threshold'] = global_thresholds['rolling_MAE_24_q95']

    records['warning_abs_q95'] = records['abs_error'] > records['abs_q95_threshold']
    records['warning_robust_z'] = records['robust_residual_z_score'].abs() > 3
    records['warning_rolling_mae_24'] = records['rolling_MAE_24'] > records['rolling_mae_24_q95_threshold']
    records['warning_flag'] = records[['warning_abs_q95', 'warning_robust_z', 'warning_rolling_mae_24']].any(axis=1)

    warning_reason = pd.Series('', index=records.index, dtype='object')
    for column, reason in [
        ('warning_abs_q95', 'abs_q95'),
        ('warning_robust_z', 'robust_z_gt_3'),
        ('warning_rolling_mae_24', 'rolling_mae_24_q95'),
    ]:
        warning_reason.loc[records[column]] = warning_reason.loc[records[column]] + reason + ';'
    records['warning_reason'] = warning_reason.str.rstrip(';')
    records.loc[records['warning_reason'].eq(''), 'warning_reason'] = 'none'
    return records


validation_residuals = construct_residual_records(val_work_df, validation_predictions, 'validation', thresholds)

if RUN_FULL_DIAGNOSTICS:
    assert_full_run_feature_availability(test_work_df, feature_columns)
    test_predictions = predict_residual_source(model_artifacts, test_work_df)
    TEST_EVALUATION_COUNT += 1
    test_residuals = construct_residual_records(test_work_df, test_predictions, 'test', thresholds)
else:
    test_residuals = None

residual_records = validation_residuals if test_residuals is None else pd.concat([validation_residuals, test_residuals], ignore_index=True)
model_fit_audit_df = build_model_fit_audit(train_work_df, val_work_df, test_work_df if RUN_FULL_DIAGNOSTICS else None, train_predictions, validation_predictions, test_predictions)

missing_residual_columns = [column for column in RESIDUAL_RECORD_COLUMNS if column not in residual_records.columns]
if missing_residual_columns:
    raise AssertionError(f'Missing residual record columns: {missing_residual_columns}')

residual_records[[PARK_COL, TIME_COL, 'split'] + RESIDUAL_RECORD_COLUMNS].head()

## Park-Level Residual Summary

In [ ]:
def summarize_by_park(records: pd.DataFrame) -> pd.DataFrame:
    return (
        records.groupby(['split', PARK_COL])
        .agg(
            rows=('abs_error', 'size'),
            MAE=('abs_error', 'mean'),
            RMSE=('squared_error', lambda s: float(math.sqrt(s.mean()))),
            mean_residual=('residual', 'mean'),
            median_residual=('residual', 'median'),
            warning_rate=('warning_flag', 'mean'),
            max_rolling_MAE_24=('rolling_MAE_24', 'max'),
        )
        .reset_index()
        .sort_values(['split', 'MAE'], ascending=[True, False])
    )


park_residual_summary_df = summarize_by_park(residual_records)
park_residual_summary_df.head(20)

## Operating-Regime Summary

In [ ]:
def candidate_regime_columns(df: pd.DataFrame) -> list[str]:
    candidates = []
    for column in df.columns:
        lower = str(column).lower()
        if column in {TARGET_COL, TIME_COL}:
            continue
        if is_prediction_like_column(column) or is_diagnostic_leakage_like_column(column):
            continue
        if pd.api.types.is_numeric_dtype(df[column]) and any(token in lower for token in ['wind', 'speed', 'temperature', 'pressure']):
            candidates.append(column)
    return candidates


def derive_operating_regime_policy(validation: pd.DataFrame) -> dict:
    candidates = candidate_regime_columns(validation)
    if not candidates:
        return {'status': 'no_suitable_regime_column', 'regime_source_column': None, 'bin_edges': []}
    regime_column = candidates[0]
    validation_values = pd.to_numeric(validation[regime_column], errors='coerce').dropna()
    if validation_values.nunique() < 2:
        return {'status': 'insufficient_validation_variation', 'regime_source_column': regime_column, 'bin_edges': []}
    _, bin_edges = pd.qcut(validation_values, q=4, duplicates='drop', retbins=True)
    bin_edges = np.unique(bin_edges)
    if len(bin_edges) < 2:
        return {'status': 'insufficient_validation_bins', 'regime_source_column': regime_column, 'bin_edges': []}
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    return {'status': 'validation_bins_ready', 'regime_source_column': regime_column, 'bin_edges': bin_edges.tolist()}


def summarize_operating_regime(records: pd.DataFrame, source_df: pd.DataFrame, policy: dict) -> pd.DataFrame:
    regime_column = policy.get('regime_source_column')
    bin_edges = policy.get('bin_edges', [])
    if not regime_column or not bin_edges or regime_column not in source_df.columns:
        return pd.DataFrame([{'status': policy.get('status', 'not_available'), 'message': 'validation-derived regime bins unavailable'}])
    aligned = records.copy()
    source_ordered = source_df.sort_values([PARK_COL, TIME_COL]).reset_index(drop=True)
    aligned[regime_column] = pd.to_numeric(source_ordered[regime_column].values[: len(aligned)], errors='coerce')
    aligned['regime_bin'] = pd.cut(aligned[regime_column], bins=bin_edges, include_lowest=True)
    return (
        aligned.groupby(['split', 'regime_bin'], observed=False)
        .agg(rows=('abs_error', 'size'), MAE=('abs_error', 'mean'), mean_residual=('residual', 'mean'), warning_rate=('warning_flag', 'mean'))
        .reset_index()
        .assign(regime_source_column=regime_column)
        .assign(regime_bin_source='validation')
    )


operating_regime_policy = derive_operating_regime_policy(val_work_df)
operating_regime_policy_df = pd.DataFrame(
    [
        {
            'status': operating_regime_policy['status'],
            'regime_source_column': operating_regime_policy.get('regime_source_column'),
            'bin_edges': json.dumps(operating_regime_policy.get('bin_edges', [])),
        }
    ]
)

validation_operating_summary_df = summarize_operating_regime(validation_residuals, val_work_df, operating_regime_policy)
if test_residuals is not None:
    test_operating_summary_df = summarize_operating_regime(test_residuals, test_work_df, operating_regime_policy)
    operating_regime_summary_df = pd.concat([validation_operating_summary_df, test_operating_summary_df], ignore_index=True)
else:
    operating_regime_summary_df = validation_operating_summary_df

operating_regime_summary_df

## Temporal Summary

In [ ]:
def summarize_temporal_patterns(records: pd.DataFrame) -> pd.DataFrame:
    temporal = records.copy()
    temporal[TIME_COL] = pd.to_datetime(temporal[TIME_COL], errors='coerce')
    temporal['hour'] = temporal[TIME_COL].dt.hour
    temporal['month'] = temporal[TIME_COL].dt.month
    return (
        temporal.groupby(['split', 'month', 'hour'], dropna=False)
        .agg(rows=('abs_error', 'size'), MAE=('abs_error', 'mean'), mean_residual=('residual', 'mean'), warning_rate=('warning_flag', 'mean'))
        .reset_index()
        .sort_values(['split', 'month', 'hour'])
    )


temporal_summary_df = summarize_temporal_patterns(residual_records)
temporal_summary_df.head(24)

## Directional Bias Summary

In [ ]:
def summarize_directional_bias(records: pd.DataFrame) -> pd.DataFrame:
    return (
        records.groupby(['split', PARK_COL])
        .agg(
            rows=('residual', 'size'),
            mean_residual=('residual', 'mean'),
            median_residual=('residual', 'median'),
            positive_residual_rate=('residual', lambda s: float((s > 0).mean())),
            negative_residual_rate=('residual', lambda s: float((s < 0).mean())),
        )
        .reset_index()
        .sort_values(['split', 'mean_residual'])
    )


directional_bias_summary_df = summarize_directional_bias(residual_records)
directional_bias_summary_df.head(20)

## Warning-Event Extraction

In [ ]:
def dominant_warning_reason(series: pd.Series) -> str:
    reasons = []
    for value in series.dropna().astype(str):
        reasons.extend([reason for reason in value.split(';') if reason and reason != 'none'])
    if not reasons:
        return 'none'
    return pd.Series(reasons).value_counts().index[0]


def extract_warning_events(records: pd.DataFrame) -> pd.DataFrame:
    ordered = records.sort_values(['split', PARK_COL, TIME_COL]).copy()
    previous_warning = ordered.groupby(['split', PARK_COL])['warning_flag'].shift(fill_value=False)
    event_start = ordered['warning_flag'] & ~previous_warning
    ordered['warning_event_id'] = event_start.groupby([ordered['split'], ordered[PARK_COL]]).cumsum()
    event_rows = ordered.loc[ordered['warning_flag']].copy()
    if event_rows.empty:
        return pd.DataFrame(
            columns=[
                'split',
                PARK_COL,
                'event_start',
                'event_end',
                'duration_rows',
                'mean_abs_error',
                'max_abs_error',
                'mean_residual',
                'max_rolling_MAE_24',
                'dominant_warning_reason',
            ]
        )
    return (
        event_rows.groupby(['split', PARK_COL, 'warning_event_id'])
        .agg(
            event_start=(TIME_COL, 'min'),
            event_end=(TIME_COL, 'max'),
            duration_rows=('warning_flag', 'size'),
            mean_abs_error=('abs_error', 'mean'),
            max_abs_error=('abs_error', 'max'),
            mean_residual=('residual', 'mean'),
            max_rolling_MAE_24=('rolling_MAE_24', 'max'),
            dominant_warning_reason=('warning_reason', dominant_warning_reason),
        )
        .reset_index()
        .drop(columns=['warning_event_id'])
        .sort_values(['split', PARK_COL, 'event_start'])
        .reset_index(drop=True)
    )


warning_events_df = extract_warning_events(residual_records)
warning_events_df.head(20)

## Residual Persistence And Autocorrelation Diagnostics

In [ ]:
def lag1_autocorrelation(series: pd.Series) -> float:
    clean = series.dropna()
    if len(clean) < 3 or clean.nunique() <= 1:
        return np.nan
    return float(clean.autocorr(lag=1))


def summarize_residual_persistence(records: pd.DataFrame) -> pd.DataFrame:
    return (
        records.sort_values(['split', PARK_COL, TIME_COL])
        .groupby(['split', PARK_COL])
        .agg(
            rows=('residual', 'size'),
            residual_lag1_autocorr=('residual', lag1_autocorrelation),
            abs_error_lag1_autocorr=('abs_error', lag1_autocorrelation),
            mean_rolling_MAE_24=('rolling_MAE_24', 'mean'),
            max_rolling_MAE_72=('rolling_MAE_72', 'max'),
            warning_rate=('warning_flag', 'mean'),
        )
        .reset_index()
    )


residual_persistence_df = summarize_residual_persistence(residual_records)
residual_persistence_df.head(20)

## Metadata And Spatial Diagnostics

In [ ]:
def metadata_spatial_columns(df: pd.DataFrame) -> list[str]:
    tokens = ['lat', 'lon', 'longitude', 'latitude', 'capacity', 'hub', 'rotor', 'turbine', 'height']
    selected = []
    for column in df.columns:
        lower = str(column).lower()
        if column in {TARGET_COL, TIME_COL, PARK_COL}:
            continue
        if any(token in lower for token in tokens):
            selected.append(column)
    return selected


def summarize_metadata_spatial(records: pd.DataFrame, source_df: pd.DataFrame) -> pd.DataFrame:
    columns = metadata_spatial_columns(source_df)
    if not columns:
        return pd.DataFrame(columns=['status', 'message'])
    metadata = source_df[[PARK_COL] + columns].drop_duplicates(subset=[PARK_COL]).copy()
    park_metrics = summarize_by_park(records).groupby(PARK_COL).agg(MAE=('MAE', 'mean'), warning_rate=('warning_rate', 'mean')).reset_index()
    return metadata.merge(park_metrics, on=PARK_COL, how='left')


if RUN_FULL_DIAGNOSTICS and test_residuals is not None:
    metadata_source_df = pd.concat([val_work_df, test_work_df], ignore_index=True)
else:
    metadata_source_df = val_work_df.copy()

metadata_spatial_summary_df = summarize_metadata_spatial(residual_records, metadata_source_df)
metadata_spatial_summary_df.head(20)

## Comparison Boundary Against NB20

NB21 compares diagnostic behavior against NB20 only after a controlled full run has been completed. This scaffold intentionally contains placeholders rather than empirical NB21 values. NB20 remains the weaker baseline-residual diagnostic reference; NB21 asks whether the same PHM-oriented warning layer persists under a stronger leakage-safe residual source.

In [ ]:
comparison_boundary_df = pd.DataFrame(
    [
        {
            'comparison_item': 'overall_residual_error',
            'NB20_reference': 'baseline-residual diagnostics',
            'NB21_value': 'pending_full_run',
            'interpretation_boundary': 'compare selectivity after full run only',
        },
        {
            'comparison_item': 'warning_rate',
            'NB20_reference': 'validation-calibrated baseline warning layer',
            'NB21_value': 'pending_full_run',
            'interpretation_boundary': 'do not infer rare fault alarms without labels',
        },
        {
            'comparison_item': 'park_level_rank_stability',
            'NB20_reference': 'NB20 park-level residual ranking',
            'NB21_value': 'pending_full_run',
            'interpretation_boundary': 'use as diagnostic evidence, not fault confirmation',
        },
    ]
)

comparison_boundary_df

## Manuscript-Safe Interpretation

NB21 can support manuscript discussion only within a forecasting-based residual diagnostics boundary. A lower-residual strong model may reduce broad false-positive behavior and highlight more selective persistent deviations, but without fault labels the warnings are not confirmed turbine faults. The appropriate claim is that validation-calibrated residual indicators identify systematic deviations from expected forecast behavior under a stronger leakage-safe residual source.

Any empirical statement comparing NB21 to NB20 must be filled only after a full controlled run and must preserve the distinction between diagnostic evidence and confirmed PHM deployment.

## Self-Checks

In [ ]:
def nb21_export_path(output_dir: Path, stem: str) -> Path:
    return output_dir / f'{EXPORT_PREFIX}_{stem}.csv'


def planned_export_paths(output_dir: Path) -> set[Path]:
    required_stems = {
        'run_manifest',
        'path_audit',
        'split_temporal_audit',
        'feature_audit',
        'preprocessing_audit',
        'model_policy',
        'model_metrics',
        'threshold_policy',
        'threshold_per_park',
        'park_level_summary',
        'warning_event_summary',
        'operating_regime_summary',
        'temporal_summary',
        'directional_bias_summary',
        'residual_persistence_summary',
        'metadata_spatial_summary',
        'comparison_boundary',
        'self_checks',
        'export_audit',
    }
    planned = {nb21_export_path(output_dir, stem) for stem in required_stems}
    if FULL_EXPORT_RESIDUAL_RECORDS:
        planned.add(nb21_export_path(output_dir, 'validation_residual_records'))
        planned.add(nb21_export_path(output_dir, 'test_residual_records'))
    return planned


def build_run_manifest() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                'notebook': '21_strong_model_residual_phm_diagnostics.ipynb',
                'SMOKE_MODE': SMOKE_MODE,
                'RUN_FULL_DIAGNOSTICS': RUN_FULL_DIAGNOSTICS,
                'EXPORT_RESULTS': EXPORT_RESULTS,
                'FULL_EXPORT_RESIDUAL_RECORDS': FULL_EXPORT_RESIDUAL_RECORDS,
                'RANDOM_STATE': RANDOM_STATE,
                'target': TARGET_COL,
                'output_dir': str(OUTPUT_DIR),
                'selected_model': model_artifacts['model_policy']['selected_model'],
            }
        ]
    )


def run_self_checks() -> pd.DataFrame:
    checks = []
    checks.append({'check': 'smoke_requires_no_full_run', 'passed': (not SMOKE_MODE) or (not RUN_FULL_DIAGNOSTICS and not EXPORT_RESULTS)})
    checks.append({'check': 'no_full_residual_export_by_default', 'passed': not FULL_EXPORT_RESIDUAL_RECORDS})
    checks.append({'check': 'test_not_used_in_default_work_subset', 'passed': RUN_FULL_DIAGNOSTICS or test_work_df.empty})
    checks.append({'check': 'no_model_checkpoint_target', 'passed': True})
    checks.append({'check': 'baseline_metrics_not_planned', 'passed': DATA_DIR / 'baseline_metrics.csv' not in planned_export_paths(OUTPUT_DIR)})
    checks.append({'check': 'requirements_not_planned', 'passed': Path('requirements.txt') not in planned_export_paths(OUTPUT_DIR)})
    checks.append({'check': 'export_dir_is_nb21_only', 'passed': OUTPUT_DIR.as_posix().endswith('data/processed/diagnostics/strong_model_residual_phm')})
    checks.append({'check': 'all_planned_exports_use_nb21_prefix', 'passed': all(path.name.startswith(f'{EXPORT_PREFIX}_') for path in planned_export_paths(OUTPUT_DIR))})
    expected_test_evaluations = 1 if RUN_FULL_DIAGNOSTICS else 0
    checks.append({'check': 'exactly_one_test_evaluation_path_when_full', 'passed': TEST_EVALUATION_COUNT == expected_test_evaluations})
    checks_df = pd.DataFrame(checks)
    if not checks_df['passed'].all():
        raise AssertionError(checks_df.loc[~checks_df['passed']].to_string(index=False))
    return checks_df


run_manifest_df = build_run_manifest()
self_checks_df = run_self_checks()
self_checks_df

## Export Cells

Exports are disabled by default. Summary artifacts are written only under `data/processed/diagnostics/strong_model_residual_phm/`. Full residual records are written only when `FULL_EXPORT_RESIDUAL_RECORDS=True`.

In [ ]:
if EXPORT_RESULTS and not RUN_FULL_DIAGNOSTICS:
    raise RuntimeError('EXPORT_RESULTS requires RUN_FULL_DIAGNOSTICS=True for NB21.')

export_audit_df = pd.DataFrame(
    [
        {
            'path': str(path),
            'will_write': EXPORT_RESULTS,
            'full_residual_record_export': FULL_EXPORT_RESIDUAL_RECORDS and 'residual_records' in path.name,
        }
        for path in sorted(planned_export_paths(OUTPUT_DIR))
    ]
)

if EXPORT_RESULTS:
    planned_paths = planned_export_paths(OUTPUT_DIR)
    prohibited_overlap = {path for path in planned_paths if path in PROHIBITED_WRITE_PATHS}
    if prohibited_overlap:
        raise AssertionError(f'Prohibited export path requested: {sorted(map(str, prohibited_overlap))}')

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    run_manifest_df.to_csv(nb21_export_path(OUTPUT_DIR, 'run_manifest'), index=False)
    path_audit_df.to_csv(nb21_export_path(OUTPUT_DIR, 'path_audit'), index=False)
    split_temporal_audit_df.merge(temporal_order_sanity_df, on='split', how='left').to_csv(nb21_export_path(OUTPUT_DIR, 'split_temporal_audit'), index=False)
    feature_schema_audit_df.to_csv(nb21_export_path(OUTPUT_DIR, 'feature_audit'), index=False)
    preprocessing_audit_df.to_csv(nb21_export_path(OUTPUT_DIR, 'preprocessing_audit'), index=False)
    model_policy_df.to_csv(nb21_export_path(OUTPUT_DIR, 'model_policy'), index=False)
    model_fit_audit_df.to_csv(nb21_export_path(OUTPUT_DIR, 'model_metrics'), index=False)
    threshold_global_df.to_csv(nb21_export_path(OUTPUT_DIR, 'threshold_policy'), index=False)
    threshold_per_park_df.to_csv(nb21_export_path(OUTPUT_DIR, 'threshold_per_park'), index=False)
    park_residual_summary_df.to_csv(nb21_export_path(OUTPUT_DIR, 'park_level_summary'), index=False)
    warning_events_df.to_csv(nb21_export_path(OUTPUT_DIR, 'warning_event_summary'), index=False)
    operating_regime_summary_df.to_csv(nb21_export_path(OUTPUT_DIR, 'operating_regime_summary'), index=False)
    temporal_summary_df.to_csv(nb21_export_path(OUTPUT_DIR, 'temporal_summary'), index=False)
    directional_bias_summary_df.to_csv(nb21_export_path(OUTPUT_DIR, 'directional_bias_summary'), index=False)
    residual_persistence_df.to_csv(nb21_export_path(OUTPUT_DIR, 'residual_persistence_summary'), index=False)
    metadata_spatial_summary_df.to_csv(nb21_export_path(OUTPUT_DIR, 'metadata_spatial_summary'), index=False)
    comparison_boundary_df.to_csv(nb21_export_path(OUTPUT_DIR, 'comparison_boundary'), index=False)
    self_checks_df.to_csv(nb21_export_path(OUTPUT_DIR, 'self_checks'), index=False)
    export_audit_df.to_csv(nb21_export_path(OUTPUT_DIR, 'export_audit'), index=False)

    if FULL_EXPORT_RESIDUAL_RECORDS:
        validation_residuals.to_csv(nb21_export_path(OUTPUT_DIR, 'validation_residual_records'), index=False)
        if test_residuals is not None:
            test_residuals.to_csv(nb21_export_path(OUTPUT_DIR, 'test_residual_records'), index=False)

export_audit_df